# 14 — Multi-Query and Grouped-Query Attention

## Goal

Standard Multi-Head Attention uses the same number of query, key, and
value heads.

If the model has $H$ attention heads,

$$
H_Q = H_K = H_V = H.
$$

During autoregressive inference, however, only keys and values need to
be stored in the KV cache.

This raises an important question:

> Do all query heads really need independent key and value heads?

This lesson studies two alternatives:

- Multi-Query Attention (MQA),
- Grouped-Query Attention (GQA).

The main goals are to understand:

1. which attention heads are actually stored in the KV cache,
2. how several query heads can share key/value heads,
3. how MHA, GQA, and MQA differ only in head organization,
4. how KV-cache memory changes,
5. and how to implement all three mechanisms explicitly.

Explain with a graph:
![MQA & GQA](./imgs/mqa_gqa.png)

In [1]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F
from einops import rearrange

from llmfp.nn.rope import (
    apply_rope,
    build_rope_cos_sin,
)

## 1. Query Heads and KV Heads

In standard Multi-Head Attention, the model dimension $C$ is divided
into $H$ heads:

$$
C = HD.
$$

Queries, keys, and values are all organized as

$$
Q,K,V
\in
\mathbb{R}^{B\times H\times T\times D}.
$$

Therefore,

$$
H_Q = H_K = H_V = H.
$$

During autoregressive decoding, queries are temporary.

Only keys and values from previous tokens need to remain available for
future queries.

The KV cache therefore stores tensors with shapes

$$
K_{\text{cache}},
V_{\text{cache}}
\in
\mathbb{R}^{B\times H_{KV}\times T\times D}.
$$

The number of KV heads directly affects inference memory.

## 2. Multi-Query Attention

Multi-Query Attention keeps multiple query heads but uses only one key
head and one value head.

If the model has $H_Q$ query heads,

$$
Q
\in
\mathbb{R}^{B \times H_Q \times T \times D},
$$

while keys and values have shapes

$$
K,V
\in
\mathbb{R}^{B \times 1 \times T \times D}.
$$

All query heads therefore attend over the same key and value
representations.

The query heads remain independent because each query head still has
its own learned query projection.

Only the key/value representations are shared.

### Projection Widths

For standard MHA,

$$
H_Q = H_{KV} = H,
$$

so query, key, and value projections all produce $HD=C$ features.

For MQA,

$$
H_{KV}=1.
$$

Therefore,

$$
W_Q:
C \rightarrow H_QD = C,
$$

but

$$
W_K:
C \rightarrow D,
$$

and

$$
W_V:
C \rightarrow D.
$$

The key and value projections are therefore much narrower than in MHA.

In [2]:
batch_size: int = 2
sequence_length: int = 5
embedding_dim: int = 8
num_query_heads: int = 4

head_dim: int = embedding_dim // num_query_heads

x: torch.Tensor = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim,
)

query_projection = nn.Linear(
    embedding_dim,
    embedding_dim,
    bias=False,
)

key_projection = nn.Linear(
    embedding_dim,
    head_dim,
    bias=False,
)

value_projection = nn.Linear(
    embedding_dim,
    head_dim,
    bias=False,
)

In [3]:
q: torch.Tensor = query_projection(x)
k: torch.Tensor = key_projection(x)
v: torch.Tensor = value_projection(x)

print("Q before heads:", q.shape)
print("K before heads:", k.shape)
print("V before heads:", v.shape)

Q before heads: torch.Size([2, 5, 8])
K before heads: torch.Size([2, 5, 2])
V before heads: torch.Size([2, 5, 2])


In [4]:
q = rearrange(
    q,
    "b t (h d) -> b h t d",
    h=num_query_heads,
)

In [5]:
# KV 1 head

k = rearrange(k, "b t d -> b 1 t d")
v = rearrange(v, "b t d -> b 1 t d")

In [6]:
print("Q:", q.shape)
print("K:", k.shape)
print("V:", v.shape)

Q: torch.Size([2, 4, 5, 2])
K: torch.Size([2, 1, 5, 2])
V: torch.Size([2, 1, 5, 2])


In [7]:
scores: torch.Tensor = q @ k.transpose(-2, -1)
print(scores.shape)

torch.Size([2, 4, 5, 5])


In [8]:
scale: float = math.sqrt(head_dim)

scores: torch.Tensor = (q @ k.transpose(-2, -1)) / scale

causal_mask: torch.Tensor = torch.tril(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool,
        device=x.device,
    )
)

scores = scores.masked_fill(
    ~causal_mask,
    float("-inf"),
)

attention_weights: torch.Tensor = F.softmax(
    scores,
    dim=-1,
)

attended: torch.Tensor = attention_weights @ v

print(attended.shape)

attended = rearrange(
    attended,
    "b h t d -> b t (h d)",
)

print(attended.shape)

torch.Size([2, 4, 5, 2])
torch.Size([2, 5, 8])


In [9]:
cos_values, sin_values = build_rope_cos_sin(
    sequence_length=sequence_length,
    head_dim=head_dim,
    device=x.device,
    dtype=q.dtype,
)

q = apply_rope(
    q,
    cos_values,
    sin_values,
)

k = apply_rope(
    k,
    cos_values,
    sin_values,
)

In [10]:
from typing import Any


class MultiQueryAttention(nn.Module):
    """Causal Multi-Query Attention with RoPE.

    Multiple independent query heads share one key head and one value
    head.

    Args:
        embedding_dim: Width of the residual stream.
        num_query_heads: Number of query heads.
        rope_base: Base controlling RoPE frequencies.
    """

    def __init__(
        self,
        embedding_dim: int,
        num_query_heads: int,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if embedding_dim % num_query_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_query_heads"
            )

        self.embedding_dim: int = embedding_dim
        self.num_query_heads: int = num_query_heads
        self.head_dim: int = embedding_dim // num_query_heads
        self.rope_base: float = rope_base

        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        # Q keeps the full model width because there are H_Q query heads.
        self.query_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )
        # MQA has only one key head and one value head.
        self.key_projection = nn.Linear(
            embedding_dim, self.head_dim, bias=False
        )
        self.value_projection = nn.Linear(
            embedding_dim, self.head_dim, bias=False
        )
        self.output_projection = nn.Linear(
            embedding_dim, embedding_dim, bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply causal Multi-Query Attention.

        Args:
            x: Residual-stream tensor with shape `(B, T, C)`.

        Returns:
            Attention output with shape `(B, T, C)`.
        """

        _, sequence_length, _ = x.shape

        q: torch.Tensor = self.query_projection(x)
        k: torch.Tensor = self.key_projection(x)
        v: torch.Tensor = self.value_projection(x)

        # Q is split into multiple independent query heads
        q = rearrange(q, "b t (h d) -> b h t d", h=self.num_query_heads)

        # K/V each contain only one shared head.
        k = rearrange(k, "b t d -> b 1 t d")
        v = rearrange(v, "b t d -> b 1 t d")

        cos_values, sin_values = build_rope_cos_sin(
            sequence_length=sequence_length,
            head_dim=self.head_dim,
            base=self.rope_base,
            dtype=q.dtype,
        )

        q = apply_rope(q, cos_values, sin_values)
        k = apply_rope(k, cos_values, sin_values)

        # The singleton KV-head axis broadcasts across all query heads.
        scores: torch.Tensor = (q @ k.transpose(-2, -1)) / math.sqrt(
            self.head_dim
        )

        causal_mask: torch.Tensor = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=x.device,
            )
        )

        scores = scores.masked_fill(~causal_mask, float("-inf"))

        attention_weights: torch.Tensor = F.softmax(scores, dim=-1)

        attended: torch.Tensor = attention_weights @ v
        attended = rearrange(
            attended, "b h t d -> b t (h d)"
        )  # (batch, head_num, time, dim) -> (batch, time, emb)

        return self.output_projection(attended)

In [11]:
attention = MultiQueryAttention(
    embedding_dim=32,
    num_query_heads=4,
)

x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

output: torch.Tensor = attention(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 10, 32])
output: torch.Size([2, 10, 32])


## 3. Grouped-Query Attention

Grouped-Query Attention lies between Multi-Head Attention and
Multi-Query Attention.

Instead of assigning one key/value head to every query head, several
query heads share one key/value head.

Let

$$
H_Q
$$

be the number of query heads and

$$
H_{KV}
$$

the number of key/value heads.

GQA requires

$$
H_Q \bmod H_{KV} = 0.
$$

The number of query heads sharing each KV head is

$$
G
=
\frac{H_Q}{H_{KV}}.
$$

For example, with

$$
H_Q=4,
\qquad
H_{KV}=2,
$$

we obtain

$$
G=2.
$$

The mapping is

<pre>
Q0 ─┐
    ├──→ K0 / V0
Q1 ─┘

Q2 ─┐
    ├──→ K1 / V1
Q3 ─┘
</pre>

### Projection Widths in GQA

The query projection still produces one representation per query head:

$$
C
\rightarrow
H_QD.
$$

Since

$$
C=H_QD,
$$

the query projection remains

$$
C\rightarrow C.
$$

Keys and values only need enough features for the KV heads:

$$
C
\rightarrow
H_{KV}D.
$$

Therefore, reducing $H_{KV}$ reduces both key/value projection size and
KV-cache size.

In [12]:
batch_size: int = 2
sequence_length: int = 5
embedding_dim: int = 8

num_query_heads: int = 4
num_kv_heads: int = 2

head_dim: int = embedding_dim // num_query_heads

queries_per_kv_head: int = num_query_heads // num_kv_heads

print("head_dim:", head_dim)
print(
    "queries per KV head:",
    queries_per_kv_head,
)

head_dim: 2
queries per KV head: 2


In [13]:
x: torch.Tensor = torch.randn(
    batch_size,
    sequence_length,
    embedding_dim,
)

query_projection = nn.Linear(
    embedding_dim,
    num_query_heads * head_dim,
    bias=False,
)

key_projection = nn.Linear(
    embedding_dim,
    num_kv_heads * head_dim,
    bias=False,
)

value_projection = nn.Linear(
    embedding_dim,
    num_kv_heads * head_dim,
    bias=False,
)

In [14]:
q: torch.Tensor = query_projection(x)
k: torch.Tensor = key_projection(x)
v: torch.Tensor = value_projection(x)

print("Q before split:", q.shape)
print("K before split:", k.shape)
print("V before split:", v.shape)

Q before split: torch.Size([2, 5, 8])
K before split: torch.Size([2, 5, 4])
V before split: torch.Size([2, 5, 4])


In [15]:
q = rearrange(
    q,
    "b t (h d) -> b h t d",
    h=num_query_heads,
)

k = rearrange(
    k,
    "b t (h d) -> b h t d",
    h=num_kv_heads,
)

v = rearrange(
    v,
    "b t (h d) -> b h t d",
    h=num_kv_heads,
)

print(q.shape)
print(k.shape)
print(v.shape)

torch.Size([2, 4, 5, 2])
torch.Size([2, 2, 5, 2])
torch.Size([2, 2, 5, 2])


In [16]:
def expand_kv_heads_manual(
    x: torch.Tensor,
    num_query_heads: int,
) -> torch.Tensor:
    """Expand KV heads to match the number of query heads.

    This educational implementation explicitly repeats each KV head for
    the query heads assigned to that group.

    Args:
        x: Key or value tensor with shape `(B, H_KV, T, D)`.
        num_query_heads: Number of query heads.

    Returns:
        Expanded tensor with shape `(B, H_Q, T, D)`.

    Raises:
        ValueError: If the query-head count is not divisible by the
            number of KV heads.
    """
    num_kv_heads: int = x.shape[1]

    if num_query_heads % num_kv_heads != 0:
        raise ValueError("num_query_heads must be divisible by num_kv_heads.")

    queries_per_kv_head: int = num_query_heads // num_kv_heads

    expanded_heads: list[torch.Tensor] = []

    for kv_head_index in range(num_kv_heads):
        # Preserve the head axis by slicing with `index:index + 1`.
        kv_head: torch.Tensor = x[
            :,
            kv_head_index : kv_head_index + 1,
            :,
            :,
        ]

        for _ in range(queries_per_kv_head):
            expanded_heads.append(kv_head)

    return torch.cat(
        expanded_heads,
        dim=1,
    )

In [17]:
k_expanded: torch.Tensor = expand_kv_heads_manual(
    k,
    num_query_heads=num_query_heads,
)

v_expanded: torch.Tensor = expand_kv_heads_manual(
    v,
    num_query_heads=num_query_heads,
)

print("Q:", q.shape)
print("K original:", k.shape)
print("K expanded:", k_expanded.shape)
print("V expanded:", v_expanded.shape)

Q: torch.Size([2, 4, 5, 2])
K original: torch.Size([2, 2, 5, 2])
K expanded: torch.Size([2, 4, 5, 2])
V expanded: torch.Size([2, 4, 5, 2])


In [18]:
scores: torch.Tensor = (
    q
    @ k_expanded.transpose(
        -2,
        -1,
    )
) / math.sqrt(head_dim)

causal_mask: torch.Tensor = torch.tril(
    torch.ones(
        sequence_length,
        sequence_length,
        dtype=torch.bool,
        device=x.device,
    )
)

scores = scores.masked_fill(
    ~causal_mask,
    float("-inf"),
)

attention_weights: torch.Tensor = F.softmax(
    scores,
    dim=-1,
)

attended: torch.Tensor = attention_weights @ v_expanded

cos_values, sin_values = build_rope_cos_sin(
    sequence_length=sequence_length,
    head_dim=head_dim,
    device=x.device,
    dtype=q.dtype,
)

q = apply_rope(
    q,
    cos_values,
    sin_values,
)

k = apply_rope(
    k,
    cos_values,
    sin_values,
)

In [19]:
class GroupedQueryAttention(nn.Module):
    """Causal Grouped-Query Attention with RoPE.

    Several query heads share each key/value head.

    Args:
        embedding_dim: Width of the residual stream.
        num_query_heads: Number of query heads.
        num_kv_heads: Number of shared key/value heads.
        rope_base: Base controlling RoPE frequencies.
    """

    def __init__(
        self,
        embedding_dim: int,
        num_query_heads: int,
        num_kv_heads: int,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if embedding_dim % num_query_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_query_heads."
            )

        if num_query_heads % num_kv_heads != 0:
            raise ValueError(
                "num_query_heads must be divisible by num_kv_heads."
            )

        self.embedding_dim: int = embedding_dim
        self.num_query_heads: int = num_query_heads
        self.num_kv_heads: int = num_kv_heads

        self.head_dim: int = embedding_dim // num_query_heads

        self.rope_base: float = rope_base

        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")

        self.query_projection = nn.Linear(
            embedding_dim,
            num_query_heads * self.head_dim,
            bias=False,
        )

        self.key_projection = nn.Linear(
            embedding_dim,
            num_kv_heads * self.head_dim,
            bias=False,
        )

        self.value_projection = nn.Linear(
            embedding_dim,
            num_kv_heads * self.head_dim,
            bias=False,
        )

        self.output_projection = nn.Linear(
            embedding_dim,
            embedding_dim,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
    ) -> torch.Tensor:
        """Apply causal Grouped-Query Attention.

        Args:
            x: Residual-stream tensor with shape `(B, T, C)`.

        Returns:
            Attention output with shape `(B, T, C)`.
        """
        _, sequence_length, _ = x.shape

        q: torch.Tensor = self.query_projection(x)

        k: torch.Tensor = self.key_projection(x)

        v: torch.Tensor = self.value_projection(x)

        q = rearrange(
            q,
            "b t (h d) -> b h t d",
            h=self.num_query_heads,
        )

        k = rearrange(
            k,
            "b t (h d) -> b h t d",
            h=self.num_kv_heads,
        )

        v = rearrange(
            v,
            "b t (h d) -> b h t d",
            h=self.num_kv_heads,
        )

        cos_values, sin_values = build_rope_cos_sin(
            sequence_length=sequence_length,
            head_dim=self.head_dim,
            base=self.rope_base,
            device=x.device,
            dtype=q.dtype,
        )

        q = apply_rope(
            q,
            cos_values,
            sin_values,
        )

        k = apply_rope(
            k,
            cos_values,
            sin_values,
        )

        # Expand only for the attention computation.
        # The compact KV representation remains `(B, H_KV, T, D)`.
        k_expanded: torch.Tensor = expand_kv_heads_manual(
            k,
            self.num_query_heads,
        )

        v_expanded: torch.Tensor = expand_kv_heads_manual(
            v,
            self.num_query_heads,
        )

        scores: torch.Tensor = (
            q
            @ k_expanded.transpose(
                -2,
                -1,
            )
        ) / math.sqrt(self.head_dim)

        causal_mask: torch.Tensor = torch.tril(
            torch.ones(
                sequence_length,
                sequence_length,
                dtype=torch.bool,
                device=x.device,
            )
        )

        scores = scores.masked_fill(
            ~causal_mask,
            float("-inf"),
        )

        attention_weights: torch.Tensor = F.softmax(
            scores,
            dim=-1,
        )

        attended: torch.Tensor = attention_weights @ v_expanded

        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )

        return self.output_projection(attended)

In [20]:
gqa = GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=2,
)

x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

output: torch.Tensor = gqa(x)

print("input:", x.shape)
print("output:", output.shape)

input: torch.Size([2, 10, 32])
output: torch.Size([2, 10, 32])


In [21]:
# MHA
GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=8,
)

# MQA
GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=1,
)

# GQA
GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=2,
)

GroupedQueryAttention(
  (query_projection): Linear(in_features=32, out_features=32, bias=False)
  (key_projection): Linear(in_features=32, out_features=8, bias=False)
  (value_projection): Linear(in_features=32, out_features=8, bias=False)
  (output_projection): Linear(in_features=32, out_features=32, bias=False)
)

## 4. Comparing MHA, GQA, and MQA

Multi-Head Attention, Grouped-Query Attention, and Multi-Query Attention
can be described using the same two head counts:

$$
H_Q
=
\text{number of query heads},
$$

and

$$
H_{KV}
=
\text{number of key/value heads}.
$$

The query-head count determines the query representation:

$$
Q
\in
\mathbb{R}^{B \times H_Q \times T \times D}.
$$

The KV-head count determines the stored key/value representations:

$$
K,V
\in
\mathbb{R}^{B \times H_{KV} \times T \times D}.
$$

The three mechanisms differ primarily in the choice of $H_{KV}$.

In [22]:
def attention_projection_parameters(
    embedding_dim: int,
    num_query_heads: int,
    num_kv_heads: int,
) -> int:
    """Count attention projection weights for a head configuration.

    Args:
        embedding_dim: Width of the residual stream.
        num_query_heads: Number of query heads.
        num_kv_heads: Number of key/value heads.

    Returns:
        Number of scalar projection parameters, ignoring biases.
    """
    if embedding_dim % num_query_heads != 0:
        raise ValueError("embedding_dim must be divisible by num_query_heads.")

    head_dim: int = embedding_dim // num_query_heads

    query_parameters: int = embedding_dim * num_query_heads * head_dim

    key_parameters: int = embedding_dim * num_kv_heads * head_dim

    value_parameters: int = key_parameters

    output_parameters: int = num_query_heads * head_dim * embedding_dim

    return (
        query_parameters + key_parameters + value_parameters + output_parameters
    )

In [23]:
embedding_dim: int = 32
num_query_heads: int = 8

mha_parameters: int = attention_projection_parameters(
    embedding_dim=embedding_dim,
    num_query_heads=num_query_heads,
    num_kv_heads=8,
)

gqa_parameters: int = attention_projection_parameters(
    embedding_dim=embedding_dim,
    num_query_heads=num_query_heads,
    num_kv_heads=2,
)

mqa_parameters: int = attention_projection_parameters(
    embedding_dim=embedding_dim,
    num_query_heads=num_query_heads,
    num_kv_heads=1,
)

print("MHA:", mha_parameters)
print("GQA:", gqa_parameters)
print("MQA:", mqa_parameters)

MHA: 4096
GQA: 2560
MQA: 2304


In [24]:
def relative_kv_cache_size(
    num_query_heads: int,
    num_kv_heads: int,
) -> float:
    """Compute KV-cache size relative to standard MHA.

    Args:
        num_query_heads: Number of query heads.
        num_kv_heads: Number of key/value heads.

    Returns:
        Relative KV-cache size, where standard MHA equals 1.0.
    """
    if num_query_heads % num_kv_heads != 0:
        raise ValueError("num_query_heads must be divisible by num_kv_heads.")

    return num_kv_heads / num_query_heads

In [25]:
for num_kv_heads in [8, 2, 1]:
    relative_size: float = relative_kv_cache_size(
        num_query_heads=8,
        num_kv_heads=num_kv_heads,
    )

    print(f"H_KV={num_kv_heads}: {relative_size:.3f} × MHA cache")

H_KV=8: 1.000 × MHA cache
H_KV=2: 0.250 × MHA cache
H_KV=1: 0.125 × MHA cache


### The Trade-off

Reducing the number of KV heads creates a trade-off.

<pre>
more KV heads
    ↓
more independent key/value representations
    ↓
larger KV cache

fewer KV heads
    ↓
more sharing between query heads
    ↓
smaller KV cache
</pre>

GQA provides an intermediate design between the representational
flexibility of MHA and the inference efficiency of MQA.

## 5. Refactoring KV-Head Expansion

The manual GQA implementation explicitly repeated each KV head with
Python loops.

For example,

<pre>
KV heads:
K0 K1

queries per KV head:
2

expanded:
K0 K0 K1 K1
</pre>

This made the query-to-KV mapping explicit.

Now that the mapping is understood, we can replace the Python loops with
a tensor operation while preserving exactly the same semantics.

In [26]:
x: torch.Tensor = torch.tensor([10, 20, 30])

repeated: torch.Tensor = torch.repeat_interleave(
    x,
    repeats=2,
)

print(repeated)

tensor([10, 10, 20, 20, 30, 30])


In [27]:
def expand_kv_heads(
    x: torch.Tensor,
    num_query_heads: int,
) -> torch.Tensor:
    """Expand KV heads to align with query heads.

    Args:
        x: Key or value tensor with shape `(B, H_KV, T, D)`.
        num_query_heads: Number of query heads.

    Returns:
        Tensor with shape `(B, H_Q, T, D)`.

    Raises:
        ValueError: If the query-head count is not divisible by the
            number of KV heads.
    """
    num_kv_heads: int = x.shape[1]

    if num_query_heads % num_kv_heads != 0:
        raise ValueError("num_query_heads must be divisible by num_kv_heads.")

    queries_per_kv_head: int = num_query_heads // num_kv_heads

    # Repeat each KV head for all query heads assigned to its group.
    return torch.repeat_interleave(
        x,
        repeats=queries_per_kv_head,
        dim=1,
    )

In [28]:
kv: torch.Tensor = torch.randn(
    2,
    2,
    5,
    4,
)

manual: torch.Tensor = expand_kv_heads_manual(
    kv,
    num_query_heads=8,
)

vectorized: torch.Tensor = expand_kv_heads(
    kv,
    num_query_heads=8,
)

print(
    "same result:",
    torch.equal(
        manual,
        vectorized,
    ),
)

print("shape:", vectorized.shape)

same result: True
shape: torch.Size([2, 8, 5, 4])


## 6. A Unified Grouped-Query Attention

After implementing MHA, MQA, and GQA separately, their common structure
is now clear.

The only architectural variable that needs to change is

$$
H_{KV}.
$$

Given

$$
H_Q,
$$

the three attention variants become:

$$
H_{KV}=H_Q
\quad\Rightarrow\quad
\text{MHA},
$$

$$
1<H_{KV}<H_Q
\quad\Rightarrow\quad
\text{GQA},
$$

and

$$
H_{KV}=1
\quad\Rightarrow\quad
\text{MQA}.
$$

This makes GQA a general formulation whose boundary cases are MHA and
MQA.

In [29]:
class GroupedQueryAttention(nn.Module):
    """Causal attention supporting MHA, GQA, and MQA.

    The number of query heads and KV heads are configured independently.

    Args:
        embedding_dim: Width of the residual stream.
        num_query_heads: Number of independent query heads.
        num_kv_heads: Number of key/value heads.
        rope_base: Base controlling RoPE frequencies.
    """

    def __init__(
        self,
        embedding_dim: int,
        num_query_heads: int,
        num_kv_heads: int,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if embedding_dim % num_query_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_query_heads"
            )

        if num_query_heads % num_kv_heads != 0:
            raise ValueError(
                "num_query_heads must be divisible by num_kv_heads"
            )

        self.embedding_dim: int = embedding_dim
        self.num_query_heads: int = num_query_heads
        self.num_kv_heads: int = num_kv_heads

        self.head_dim: int = embedding_dim // num_query_heads

        self.rope_base: float = rope_base

        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE")

        # Queries retain the full model width.
        self.query_projection = nn.Linear(
            embedding_dim, num_query_heads * self.head_dim, bias=False
        )

        # KV width depends only on the number of KV heads
        self.key_projection = nn.Linear(
            embedding_dim, num_kv_heads * self.head_dim, bias=False
        )
        self.value_projection = nn.Linear(
            embedding_dim, num_kv_heads * self.head_dim, bias=False
        )
        self.output_projection = nn.Linear(
            num_query_heads * self.head_dim, embedding_dim, bias=False
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        """Apply grouped-query causal self-attention.

        Args:
            x: Residual-stream tensor with shape `(B, T, C)`.

        Returns:
            Attention output with shape `(B, T, C)`.
        """
        _, sequence_length, _ = x.shape

        q: torch.Tensor = self.query_projection(x)
        k: torch.Tensor = self.key_projection(x)
        v: torch.Tensor = self.value_projection(x)

        # Q and KV prjecctions contain different numbers of heads

        q = rearrange(q, "b t (h d) -> b h t d", h=self.num_query_heads)
        k = rearrange(k, "b t (h d) -> b h t d", h=self.num_kv_heads)
        v = rearrange(v, "b t (h d) -> b h t d", h=self.num_kv_heads)

        cos_values, sin_values = build_rope_cos_sin(
            sequence_length=sequence_length,
            head_dim=self.head_dim,
            base=self.rope_base,
            device=x.device,
            dtype=q.dtype,
        )

        q = apply_rope(q, cos_values, sin_values)
        k = apply_rope(k, cos_values, sin_values)

        # Align compact KV heads with their assigned query-head groups.

        k_expanded: torch.Tensor = expand_kv_heads(
            k, num_query_heads=self.num_query_heads
        )
        v_expanded: torch.Tensor = expand_kv_heads(
            v, num_query_heads=self.num_query_heads
        )
        # Scaled dot-product attention itself is already understood,
        # so use PyTorch's optimized attention primitive.
        attended: torch.Tensor = F.scaled_dot_product_attention(
            q,
            k_expanded,
            v_expanded,
            is_causal=True,
        )

        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )

        return self.output_projection(attended)

In [30]:
mha = GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=8,
)

gqa = GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=2,
)
mqa = GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=1,
)
x: torch.Tensor = torch.randn(
    2,
    10,
    32,
)

for name, attention in [
    ("MHA", mha),
    ("GQA", gqa),
    ("MQA", mqa),
]:
    output: torch.Tensor = attention(x)

    print(
        f"{name}:",
        output.shape,
    )

MHA: torch.Size([2, 10, 32])
GQA: torch.Size([2, 10, 32])
MQA: torch.Size([2, 10, 32])


## 7. KV Caching with Grouped-Query Attention

The motivation for MQA and GQA comes primarily from autoregressive
inference.

The important rule is:

> Store only the compact KV representation.

For GQA,

$$
K_{\text{cache}},V_{\text{cache}}
\in
\mathbb{R}^{B\times H_{KV}\times T\times D}.
$$

The KV heads may be expanded temporarily when computing attention with
$H_Q$ query heads, but the expanded representation should not be stored
in the cache.

Therefore:

<pre>
stored cache
(B, H_KV, T, D)
        ↓
temporary head expansion
(B, H_Q, T, D)
        ↓
attention computation
</pre>

This preserves the memory advantage of MQA and GQA.

In [31]:
def expand_kv_heads(
    x: torch.Tensor,
    num_query_heads: int,
) -> torch.Tensor:
    """Expand KV heads to align with query heads.

    Each KV head is repeated for all query heads assigned to its group.

    For example:

        H_Q = 8
        H_KV = 2

    produces the logical mapping:

        KV0 → Q0, Q1, Q2, Q3
        KV1 → Q4, Q5, Q6, Q7

    Args:
        x: Key or value tensor with shape `(B, H_KV, T, D)`.
        num_query_heads: Number of query heads.

    Returns:
        Expanded tensor with shape `(B, H_Q, T, D)`.

    Raises:
        ValueError: If `num_query_heads` is not divisible by the
            number of KV heads.
    """
    num_kv_heads: int = x.shape[1]

    if num_query_heads % num_kv_heads != 0:
        raise ValueError("num_query_heads must be divisible by num_kv_heads.")

    queries_per_kv_head: int = num_query_heads // num_kv_heads

    # Repeat each KV head for every query head belonging to its group.
    return torch.repeat_interleave(
        x,
        repeats=queries_per_kv_head,
        dim=1,
    )


class GroupedQueryAttention(nn.Module):
    """Causal attention supporting MHA, GQA, and MQA with KV caching.

    The number of query heads and key/value heads are configured
    independently.

    The same class represents:

        MHA:
            num_kv_heads == num_query_heads

        GQA:
            1 < num_kv_heads < num_query_heads

        MQA:
            num_kv_heads == 1

    Args:
        embedding_dim: Width of the residual stream.
        num_query_heads: Number of independent query heads.
        num_kv_heads: Number of key/value heads.
        rope_base: Base controlling RoPE frequencies.
    """

    def __init__(
        self,
        embedding_dim: int,
        num_query_heads: int,
        num_kv_heads: int,
        rope_base: float = 10000.0,
    ) -> None:
        super().__init__()

        if embedding_dim % num_query_heads != 0:
            raise ValueError(
                "embedding_dim must be divisible by num_query_heads."
            )

        if num_query_heads % num_kv_heads != 0:
            raise ValueError(
                "num_query_heads must be divisible by num_kv_heads."
            )

        self.embedding_dim: int = embedding_dim
        self.num_query_heads: int = num_query_heads
        self.num_kv_heads: int = num_kv_heads

        self.head_dim: int = embedding_dim // num_query_heads

        self.rope_base: float = rope_base

        if self.head_dim % 2 != 0:
            raise ValueError("head_dim must be even for RoPE.")

        # Queries retain the full model width:
        #
        # C → H_Q × D = C
        self.query_projection = nn.Linear(
            embedding_dim,
            num_query_heads * self.head_dim,
            bias=False,
        )

        # Keys and values only need enough features for H_KV heads:
        #
        # C → H_KV × D
        self.key_projection = nn.Linear(
            embedding_dim,
            num_kv_heads * self.head_dim,
            bias=False,
        )

        self.value_projection = nn.Linear(
            embedding_dim,
            num_kv_heads * self.head_dim,
            bias=False,
        )

        # The attention result still contains H_Q heads, so its total
        # width is H_Q × D = C.
        self.output_projection = nn.Linear(
            num_query_heads * self.head_dim,
            embedding_dim,
            bias=False,
        )

    def forward(
        self,
        x: torch.Tensor,
        past_key: torch.Tensor | None = None,
        past_value: torch.Tensor | None = None,
        use_cache: bool = False,
    ) -> (
        torch.Tensor
        | tuple[
            torch.Tensor,
            torch.Tensor,
            torch.Tensor,
        ]
    ):
        """Apply grouped-query causal self-attention.

        Args:
            x: New input representations with shape `(B, T_new, C)`.
            past_key: Compact cached keys with shape
                `(B, H_KV, T_past, D)`, or `None`.
            past_value: Compact cached values with shape
                `(B, H_KV, T_past, D)`, or `None`.
            use_cache: Whether the updated compact KV cache should be
                returned.

        Returns:
            If `use_cache` is False:
                Attention output with shape `(B, T_new, C)`.

            If `use_cache` is True:
                A tuple containing:

                - attention output `(B, T_new, C)`,
                - key cache `(B, H_KV, T_total, D)`,
                - value cache `(B, H_KV, T_total, D)`.

        Raises:
            ValueError: If input or cache shapes are inconsistent.
        """
        if x.ndim != 3:
            raise ValueError("x must have shape (B, T, C).")

        batch_size, sequence_length, embedding_dim = x.shape

        if embedding_dim != self.embedding_dim:
            raise ValueError(
                "Input feature dimension must match embedding_dim."
            )

        # Either both cache tensors exist, or neither exists.
        if (past_key is None) != (past_value is None):
            raise ValueError(
                "past_key and past_value must both be provided or both be None."
            )

        if past_key is None:
            past_length: int = 0

        else:
            if not use_cache:
                raise ValueError(
                    "use_cache must be True when a past cache is provided."
                )

            if past_value is None:
                raise ValueError("past_value must be provided with past_key.")

            if past_key.ndim != 4:
                raise ValueError(
                    "past_key must have shape (B, H_KV, T_past, D)."
                )

            if past_key.shape != past_value.shape:
                raise ValueError(
                    "past_key and past_value must have identical shapes."
                )

            if past_key.shape[0] != batch_size:
                raise ValueError(
                    "Cache batch size must match input batch size."
                )

            if past_key.shape[1] != self.num_kv_heads:
                raise ValueError("Cache head count must match num_kv_heads.")

            if past_key.shape[-1] != self.head_dim:
                raise ValueError("Cache head dimension must match head_dim.")

            # This educational cached implementation decodes exactly
            # one new token at a time.
            if sequence_length != 1:
                raise ValueError(
                    "Cached decoding currently expects one new token."
                )

            past_length = past_key.shape[2]

        # ------------------------------------------------------------
        # 1. Q / K / V projections
        # ------------------------------------------------------------

        q: torch.Tensor = self.query_projection(x)
        k: torch.Tensor = self.key_projection(x)
        v: torch.Tensor = self.value_projection(x)

        # Queries have H_Q heads.
        #
        # (B, T_new, H_Q × D)
        #          →
        # (B, H_Q, T_new, D)
        q = rearrange(
            q,
            "b t (h d) -> b h t d",
            h=self.num_query_heads,
        )

        # Keys and values only have H_KV heads.
        #
        # (B, T_new, H_KV × D)
        #          →
        # (B, H_KV, T_new, D)
        k = rearrange(
            k,
            "b t (h d) -> b h t d",
            h=self.num_kv_heads,
        )

        v = rearrange(
            v,
            "b t (h d) -> b h t d",
            h=self.num_kv_heads,
        )

        # ------------------------------------------------------------
        # 2. RoPE
        # ------------------------------------------------------------

        # Cached decoding starts after every token already stored
        # in the KV cache.
        cos_values, sin_values = build_rope_cos_sin(
            sequence_length=sequence_length,
            head_dim=self.head_dim,
            position_offset=past_length,
            base=self.rope_base,
            device=x.device,
            dtype=q.dtype,
        )

        # Position information enters Q and K, but not V.
        q = apply_rope(
            q,
            cos_values,
            sin_values,
        )

        k = apply_rope(
            k,
            cos_values,
            sin_values,
        )

        # ------------------------------------------------------------
        # 3. Build the compact KV cache
        # ------------------------------------------------------------

        if past_key is None:
            # Training or prompt prefill.
            key_cache: torch.Tensor = k
            value_cache: torch.Tensor = v

        else:
            # Decode: append only the newly computed K/V.
            #
            # (B, H_KV, T_past, D)
            #          +
            # (B, H_KV, 1, D)
            #          →
            # (B, H_KV, T_past + 1, D)
            key_cache = torch.cat(
                [past_key, k],
                dim=2,
            )

            value_cache = torch.cat(
                [past_value, v],
                dim=2,
            )

        # ------------------------------------------------------------
        # 4. Expand KV heads only for attention computation
        # ------------------------------------------------------------

        # IMPORTANT:
        #
        # The stored cache remains compact:
        #
        #     (B, H_KV, T, D)
        #
        # Expansion only creates the logical Q-to-KV head mapping needed
        # for the attention computation.
        k_expanded: torch.Tensor = expand_kv_heads(
            key_cache,
            num_query_heads=self.num_query_heads,
        )

        v_expanded: torch.Tensor = expand_kv_heads(
            value_cache,
            num_query_heads=self.num_query_heads,
        )

        # Shapes are now aligned:
        #
        # Q:
        # (B, H_Q, T_new, D)
        #
        # K/V expanded:
        # (B, H_Q, T_total, D)

        # ------------------------------------------------------------
        # 5. Scaled dot-product attention
        # ------------------------------------------------------------

        if past_key is None:
            # During full-sequence processing, future positions exist
            # inside the same tensor and therefore require masking.
            is_causal: bool = True
        else:
            # During one-token decoding, the cache contains only past
            # tokens plus the current token. No future keys exist.
            is_causal = False

        attended: torch.Tensor = F.scaled_dot_product_attention(
            q,
            k_expanded,
            v_expanded,
            is_causal=is_causal,
        )

        # attended:
        #
        # (B, H_Q, T_new, D)

        # ------------------------------------------------------------
        # 6. Merge query heads
        # ------------------------------------------------------------

        attended = rearrange(
            attended,
            "b h t d -> b t (h d)",
        )

        # (B, H_Q, T_new, D)
        #          →
        # (B, T_new, H_Q × D)
        #          =
        # (B, T_new, C)

        output: torch.Tensor = self.output_projection(attended)

        if use_cache:
            return (
                output,
                key_cache,
                value_cache,
            )

        return output

In [33]:
gqa = GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=2,
)

prompt: torch.Tensor = torch.randn(
    1,
    5,
    32,
)

gqa.eval()

with torch.no_grad():
    (
        prompt_output,
        key_cache,
        value_cache,
    ) = gqa(
        prompt,
        use_cache=True,
    )

print("output:", prompt_output.shape)
print("K cache:", key_cache.shape)
print("V cache:", value_cache.shape)

output: torch.Size([1, 5, 32])
K cache: torch.Size([1, 2, 5, 4])
V cache: torch.Size([1, 2, 5, 4])


In [34]:
for name, num_kv_heads in [
    ("MHA", 8),
    ("GQA", 2),
    ("MQA", 1),
]:
    attention = GroupedQueryAttention(
        embedding_dim=32,
        num_query_heads=8,
        num_kv_heads=num_kv_heads,
    )

    with torch.no_grad():
        (
            _,
            key_cache,
            value_cache,
        ) = attention(
            prompt,
            use_cache=True,
        )

    print(
        f"{name:3s} cache:",
        key_cache.shape,
    )

MHA cache: torch.Size([1, 8, 5, 4])
GQA cache: torch.Size([1, 2, 5, 4])
MQA cache: torch.Size([1, 1, 5, 4])


## 8. Verifying Cached Decoding Across MHA, GQA, and MQA

The unified `GroupedQueryAttention` implementation should behave
identically whether a sequence is processed all at once or through
prefill followed by cached decoding.

Consider a sequence

<pre>
A B C D
</pre>

The output for `D` can be computed in two ways.

### Full forward

<pre>
A B C D
      ↓
attention
      ↓
output for D
</pre>

### Cached forward

<pre>
A B C
  ↓
prefill
  ↓
compact KV cache

D
↓
cached decode
↓
output for D
</pre>

The two outputs should match up to floating-point error.

We perform this test for:

- MHA: $H_{KV}=H_Q$,
- GQA: $1<H_{KV}<H_Q$,
- MQA: $H_{KV}=1$.

In [35]:
gqa = GroupedQueryAttention(
    embedding_dim=32,
    num_query_heads=8,
    num_kv_heads=2,
)

gqa.eval()

x_full: torch.Tensor = torch.randn(
    1,
    4,
    32,
)

x_prompt: torch.Tensor = x_full[:, :3, :]

x_new: torch.Tensor = x_full[:, 3:4, :]

print("full:", x_full.shape)
print("prompt:", x_prompt.shape)
print("new:", x_new.shape)

full: torch.Size([1, 4, 32])
prompt: torch.Size([1, 3, 32])
new: torch.Size([1, 1, 32])


In [36]:
with torch.no_grad():
    full_output: torch.Tensor = gqa(x_full)

full_last_output: torch.Tensor = full_output[:, -1:, :]

print(
    "full last output:",
    full_last_output.shape,
)

full last output: torch.Size([1, 1, 32])


In [38]:
with torch.no_grad():
    (
        prompt_output,
        key_cache,
        value_cache,
    ) = gqa(
        x_prompt,
        use_cache=True,
    )

print(
    "prompt output:",
    prompt_output.shape,
)

print(
    "key cache:",
    key_cache.shape,
)

print(
    "value cache:",
    value_cache.shape,
)

prompt output: torch.Size([1, 3, 32])
key cache: torch.Size([1, 2, 3, 4])
value cache: torch.Size([1, 2, 3, 4])


In [39]:
# decoding
with torch.no_grad():
    (
        cached_output,
        updated_key_cache,
        updated_value_cache,
    ) = gqa(
        x_new,
        past_key=key_cache,
        past_value=value_cache,
        use_cache=True,
    )

print(
    "cached output:",
    cached_output.shape,
)

print(
    "updated key cache:",
    updated_key_cache.shape,
)

cached output: torch.Size([1, 1, 32])
updated key cache: torch.Size([1, 2, 4, 4])


In [40]:
outputs_match: bool = torch.allclose(
    full_last_output,
    cached_output,
    atol=1e-5,
)

maximum_difference: float = (
    (full_last_output - cached_output).abs().max().item()
)

print(
    "outputs match:",
    outputs_match,
)

print(
    "maximum absolute difference:",
    maximum_difference,
)

outputs match: True
maximum absolute difference: 7.28759914636612e-08


In [41]:
def verify_cached_attention(
    *,
    embedding_dim: int,
    num_query_heads: int,
    num_kv_heads: int,
    sequence_length: int = 4,
) -> tuple[bool, float, tuple[int, ...]]:
    """Compare full attention with prefill plus cached decoding.

    Args:
        embedding_dim: Width of the residual stream.
        num_query_heads: Number of query heads.
        num_kv_heads: Number of key/value heads.
        sequence_length: Length of the complete test sequence.

    Returns:
        A tuple containing:
        - whether the outputs match,
        - the maximum absolute numerical difference,
        - the compact key-cache shape after prefill.
    """
    if sequence_length < 2:
        raise ValueError("sequence_length must be at least 2.")

    attention = GroupedQueryAttention(
        embedding_dim=embedding_dim,
        num_query_heads=num_query_heads,
        num_kv_heads=num_kv_heads,
    )

    attention.eval()

    x_full: torch.Tensor = torch.randn(
        1,
        sequence_length,
        embedding_dim,
    )

    x_prompt: torch.Tensor = x_full[:, :-1, :]

    x_new: torch.Tensor = x_full[:, -1:, :]

    with torch.no_grad():
        # Reference computation using the full sequence.
        full_output: torch.Tensor = attention(x_full)

        full_last_output: torch.Tensor = full_output[:, -1:, :]

        # Prefill creates the compact K/V cache.
        (
            _,
            key_cache,
            value_cache,
        ) = attention(
            x_prompt,
            use_cache=True,
        )

        prefill_cache_shape: tuple[int, ...] = tuple(key_cache.shape)

        # Decode only the final token using cached history.
        (
            cached_output,
            _,
            _,
        ) = attention(
            x_new,
            past_key=key_cache,
            past_value=value_cache,
            use_cache=True,
        )

    outputs_match: bool = torch.allclose(
        full_last_output,
        cached_output,
        atol=1e-5,
    )

    maximum_difference: float = (
        (full_last_output - cached_output).abs().max().item()
    )

    return (
        outputs_match,
        maximum_difference,
        prefill_cache_shape,
    )

In [42]:
configurations = [
    ("MHA", 8),
    ("GQA", 2),
    ("MQA", 1),
]

for name, num_kv_heads in configurations:
    (
        outputs_match,
        maximum_difference,
        cache_shape,
    ) = verify_cached_attention(
        embedding_dim=32,
        num_query_heads=8,
        num_kv_heads=num_kv_heads,
    )

    print(f"{name}:")

    print(
        "  outputs match:",
        outputs_match,
    )

    print(
        "  max difference:",
        maximum_difference,
    )

    print(
        "  prefill K cache:",
        cache_shape,
    )

MHA:
  outputs match: True
  max difference: 1.1920928955078125e-07
  prefill K cache: (1, 8, 3, 4)
GQA:
  outputs match: True
  max difference: 5.960464477539063e-08
  prefill K cache: (1, 2, 3, 4)
MQA:
  outputs match: True
  max difference: 5.21540641784668e-08
  prefill K cache: (1, 1, 3, 4)


### What This Test Verifies

The same `GroupedQueryAttention` implementation correctly represents all
three attention variants.

For every configuration,

$$
\text{full forward}
\approx
\text{prefill + cached decode}.
$$

At the same time, the stored KV-cache head dimension changes:

$$
H_{KV}=H_Q
\quad\text{for MHA},
$$

$$
1<H_{KV}<H_Q
\quad\text{for GQA},
$$

and

$$
H_{KV}=1
\quad\text{for MQA}.
$$

The external Transformer interface remains unchanged:

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

The optimization occurs inside the attention mechanism and, most
importantly, inside the KV cache.

## 9. Final Comparison: MHA vs GQA vs MQA

MHA, GQA, and MQA use the same query-head structure.

The main architectural difference is the number of key/value heads:

$$
H_{KV}.
$$

For a fixed number of query heads $H_Q$,

$$
H_{KV}=H_Q
$$

gives Multi-Head Attention,

$$
1 < H_{KV} < H_Q
$$

gives Grouped-Query Attention,

and

$$
H_{KV}=1
$$

gives Multi-Query Attention.

The external Transformer interface remains unchanged:

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

The differences appear internally in the key/value projections and the
KV cache.

### The Design Trade-off

Reducing $H_{KV}$ improves inference efficiency, but also increases
sharing between query heads.

MHA gives every query head an independent key/value representation.

MQA shares one key/value representation across all query heads.

GQA provides a middle ground:

<pre>
MHA
more KV diversity
more cache memory

        ↓

GQA
intermediate sharing
intermediate cache memory

        ↓

MQA
maximum KV sharing
minimum cache memory
</pre>

### One Unified View

For all three mechanisms,

$$
Q
\in
\mathbb{R}^{B\times H_Q\times T\times D}.
$$

Only the number of KV heads changes:

$$
K,V
\in
\mathbb{R}^{B\times H_{KV}\times T\times D}.
$$

The grouping factor is

$$
G
=
\frac{H_Q}{H_{KV}}.
$$

Therefore:

$$
G=1
\Rightarrow
\text{MHA},
$$

$$
1<G<H_Q
\Rightarrow
\text{GQA},
$$

and

$$
G=H_Q
\Rightarrow
\text{MQA}.
$$

The central engineering benefit comes from reducing

$$
H_{KV},
$$

because this directly reduces both K/V projection width and KV-cache
memory.

## Takeaways

This lesson generalized multi-head attention by separating two quantities
that standard MHA normally keeps equal:

$$
H_Q
=
\text{number of query heads},
$$

and

$$
H_{KV}
=
\text{number of key/value heads}.
$$

This creates one unified view of three attention mechanisms.

### MHA, GQA, and MQA differ mainly in the number of KV heads

For all three mechanisms,

$$
Q
\in
\mathbb{R}^{B\times H_Q\times T\times D}.
$$

Keys and values instead use

$$
K,V
\in
\mathbb{R}^{B\times H_{KV}\times T\times D}.
$$

The three variants are boundary cases of the same design:

$$
H_{KV}=H_Q
\Rightarrow
\text{MHA},
$$

$$
1<H_{KV}<H_Q
\Rightarrow
\text{GQA},
$$

and

$$
H_{KV}=1
\Rightarrow
\text{MQA}.
$$

The number of query heads sharing each KV head is

$$
G
=
\frac{H_Q}{H_{KV}}.
$$

Therefore, changing attention type does not require changing the external
Transformer interface.

All three still implement

$$
(B,T,C)
\rightarrow
(B,T,C).
$$

### Query diversity and KV diversity are different things

Reducing the number of KV heads does not reduce the number of query
heads.

Multiple query heads can still learn different query representations and
therefore produce different attention patterns, even when they share the
same keys and values.

This distinction is central:

<pre>
H_Q
→ query diversity

H_KV
→ key/value diversity
→ KV-cache size
</pre>

MQA keeps many independent queries while sharing one key/value
representation across all of them.

GQA provides an intermediate amount of sharing.

### Projection width depends on H_KV

The query projection produces

$$
H_QD=C
$$

features, so it remains a full-width projection:

$$
C\rightarrow C.
$$

The key and value projections only need

$$
H_{KV}D
$$

features:

$$
C\rightarrow H_{KV}D.
$$

The total projection parameter count is therefore

$$
P_{\text{attention}}
=
2C^2
+
2CH_{KV}D.
$$

Since

$$
D=\frac{C}{H_Q},
$$

this becomes

$$
P_{\text{attention}}
=
2C^2
\left(
1+\frac{H_{KV}}{H_Q}
\right).
$$

Reducing the number of KV heads therefore reduces both K/V projection
parameters and computation.

### The major inference benefit appears in the KV cache

The persistent cache stores

$$
K_{\text{cache}},
V_{\text{cache}}
\in
\mathbb{R}^{B\times H_{KV}\times T\times D}.
$$

Its memory cost across $L$ Transformer layers is proportional to

$$
2LBH_{KV}TD.
$$

Therefore,

$$
M_{\text{KV}}
\propto H_{KV}.
$$

For example, with

$$
H_Q=8,
$$

we obtain:

<pre>
MHA: H_KV = 8  → 1.000 × cache

GQA: H_KV = 2  → 0.250 × cache

MQA: H_KV = 1  → 0.125 × cache
</pre>

This is the main systems motivation for MQA and GQA.

### KV-head expansion is a compute operation, not a storage decision

For the educational GQA implementation, compact KV heads are temporarily
expanded so that they align with query heads:

$$
(B,H_{KV},T,D)
\rightarrow
(B,H_Q,T,D).
$$

However, the persistent cache always remains compact:

$$
(B,H_{KV},T,D).
$$

The expansion exists only for the attention computation.

This distinction is important:

> temporary compute representation is not the same thing as persistent
> cache representation.

### Cached decoding preserves model semantics

For MHA, GQA, and MQA, we verified that

$$
\text{full forward}
\approx
\text{prefill + cached decode}
$$

up to floating-point error.

Changing the number of KV heads changes the architecture and its
representational trade-off, but KV caching itself remains only an
inference optimization.

### One implementation can represent all three mechanisms

After implementing MHA, MQA, and GQA separately, their duplication became
clear.

A unified `GroupedQueryAttention` can represent all three simply by
changing

$$
H_{KV}.
$$

This follows the project's general learning pattern:

<pre>
implement separately
        ↓
understand the difference
        ↓
identify the shared structure
        ↓
refactor
</pre>

The central lesson is:

> Multiple query heads do not require an equal number of key/value heads.

This decoupling allows a Transformer to preserve rich query behavior
while trading some KV representational diversity for substantially lower
inference memory.